# Interactive Plotting

The forecast-tools package provides the `plotting` module.  

The main function, `plot_time_series`, creates interactive visualizations of univariate time series data with options to display training data, test data, point forecasts, and prediction intervals at various confidence levels.

**Features**

- Interactive Plotly-based visualizations
- Support for displaying training and test data
- Visualization of point forecasts
- Display of prediction intervals at multiple confidence levels
- Customizable colors and styling
- Hover functionality with vertical guide lines
- Multiple predefined color schemes

## 1. Imports

In [1]:
# standard imports
import pandas as pd

In [2]:
# forecats-tools imports
from forecast_tools.plotting import plot_time_series
from forecast_tools.datasets import load_emergency_dept
from forecast_tools.baseline import SNaive

## 2. Load data and plot data

In [3]:
HOLDOUT = 7
PERIOD = 7

attends = load_emergency_dept()

In [4]:
# train-test split
fig = plot_time_series(training_data=attends)

In [5]:
# train-test split
train, test = attends[:-HOLDOUT], attends[-HOLDOUT:]

fig = plot_time_series(
    training_data=train,
    test_data=test,
    test_data_mode="lines",
    y_axis_label="ED Attendances"
    )

## 3. Display a point forecast and prediction intervals

As an example we will create a seasonal naive model and predict 7 days ahead.  We will display the point forecast and the 80% and 95% prediction intervals. 


**Some key parameters:**

* `forecast` accepts a `pandas.DataFrame` with a `datetimeindex` (the code below illustrates how to create one if needed.)
* `prediction_intervals` accepts a `dict` that can contain multiple prediction intervals represented as `pandas.DataFrame` with a `datetimeindex`. The interval dataframes should include 2 columns labelled "lower" and "upper".
* `color_scheme` accepts "red" (default), "blue" or "green".
* `forecast_line_style` accepts a string from 'dash', 'solid', 'dot', 'dashdot'

In [6]:
model = SNaive(PERIOD)

# returns 80 and 90% prediction intervals by default.
preds, intervals = model.fit_predict(train, HOLDOUT, return_predict_int=True, alpha=(0.2, 0.05))

# convert numpy arrays to dataframe.
preds = pd.DataFrame(preds, index=test.index)
interval_dict = {
    "80% PI": pd.DataFrame(intervals[0], index=test.index, columns=["lower", "upper"]),
    "95% PI": pd.DataFrame(intervals[1], index=test.index, columns=["lower", "upper"])
}

In [7]:
fig = plot_time_series(
    training_data=train[-28:],
    test_data=test, 
    forecast=preds,
    prediction_intervals=interval_dict,
    y_axis_label="ED Attendances",
    )

In [8]:
# change colour scheme to blue
fig = plot_time_series(
    training_data=train[-28:],
    test_data=test, 
    forecast=preds,
    prediction_intervals=interval_dict,
    y_axis_label="ED Attendances",
    color_scheme="blue",
    forecast_line_style="solid"
    )

In [9]:
# change colour scheme to green
fig = plot_time_series(
    training_data=train[-28:],
    test_data=test, 
    forecast=preds,
    prediction_intervals=interval_dict,
    y_axis_label="ED Attendances",
    color_scheme="green"
    )

## 4. Common usage mistakes

The main mistake is to pass in training, test and forecast data that are not a `pd.DataFrame` or do not have a `datetimeindex`. The function has validation and will raise an error if this is attempted. For example:

In [10]:
model = SNaive(PERIOD)

# preds is returned as a numpy array
preds, intervals = model.fit_predict(train, HOLDOUT, return_predict_int=True, alpha=(0.2, 0.05))


try: 
    # pred is passed in as a numpy array and not a dataframe
    # this will raise a TypeError
    fig = plot_time_series(
        training_data=train[-28:],
        test_data=test, 
        forecast=preds,
        y_axis_label="ED Attendances",
        )
    
# catch the TypeError for the example
except TypeError as e:
    print(f"TypeError: {e}")

TypeError: forecast must be a pandas DataFrame


In [11]:
model = SNaive(PERIOD)

# returns 80 and 90% prediction intervals by default.
preds, intervals = model.fit_predict(train, HOLDOUT, return_predict_int=True, alpha=(0.2, 0.05))

# convert numpy arrays to dataframe, but forget to set the datetimeindex
preds = pd.DataFrame(preds)

try: 
    # pred is passed in as a numpy array and not a dataframe
    # this will raise a TypeError
    fig = plot_time_series(
        training_data=train[-28:],
        test_data=test, 
        forecast=preds,
        y_axis_label="ED Attendances",
        )
    
# catch the TypeError for the example
except TypeError as e:
    print(f"TypeError: {e}")

TypeError: forecast must have a DatetimeIndex
